In [ ]:
#-------------------------------------------------------------------------------
# Name:        04_Download_GEE_images
# Purpose:     Download the S2 clip under the grid
#
# Author:      Gijs van den Dool & Meggison Oritsemisan
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/19wF5gdNnmX4Lwj1kHxltKGRF6DT1yvQ7

# GEE Images:
# Output from the selection of SB areas in a 5120x5120 bounding box, retaining
# only the bounding box

# Image Patches:
# Input for the modelling, as a input for predictions

# Disclaimer: most code was developed by caleb, to work on the whole tile, the
# modification is that it performs the clip on the bounding area of the grid
# and exports in a 128x128 patch


In [ ]:
 # reminder that if you are installing libraries in a Google Colab instance you
 # will be prompted to restart your kernal

 # check packages installed:
packages = !pip list

if "mapclassify" in packages:
  print("package available")
else:
  # for visualisation of a geopandas dataframe
  !pip install mapclassify -q


if "rasterio" in packages:
  print("package available")
else:
  # for visualisation of a geopandas dataframe
  !pip install rasterio -q


#
# # alternative tests:
# try:
#     import geemap, ee
# except ModuleNotFoundError:
#     if 'google.colab' in str(get_ipython()):
#         print("package not found, installing w/ pip in Google Colab...")
#         !pip install geemap
#     else:
#         print("package not found, installing w/ conda...")
#         !conda install mamba -c conda-forge -y
#         !mamba install geemap -c conda-forge -y
#     import geemap, ee

In [ ]:
import os

import ee
import geemap
import geemap.foliumap as emap

import datetime

import json

from shapely.geometry import Point,LineString, Polygon

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import geopandas as gpd

from matplotlib import pyplot as plt

from datetime import datetime as DT
from tqdm.notebook import tqdm_notebook

/Users/misanmeggison/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
try:
        ee.Initialize()
except Exception as e:
        ee.Authenticate()
        ee.Initialize()


In [ ]:
# change these to match your configuration:
project_root = "/content/drive/MyDrive/Project Canopy Official Folder/Task 02 Data Preprocessing/Working Files/Experiment_SB"
task_folder = "02_WorkingFiles"
file_name = "ImagePatch_SB_10.geojson" # might be different for other areas

# Construct the file path using os.path.join()
file_path = os.path.join(project_root, task_folder,file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
    print("File exists.")
else:
    print("File does not exist or is not accessible.")

File does not exist or is not accessible.


In [ ]:
 # Define the file path to the shapefile
task_folder = "03_Output_Data"

# Create main output folder:
out_folder = os.path.join(project_root, task_folder, "02_download_Sentinel2_GEE/")

if os.path.exists(out_folder) == False:
  os.makedirs(out_folder)

  # Create output folder SR:
  out_folder1 = os.path.join(out_folder, "ImagePatch_SR/")
  os.makedirs(out_folder1)

  # Create output folder HM:
  out_folder2 = os.path.join(out_folder, "ImagePatch_HM/")
  os.makedirs(out_folder2)

else:
  print("folder already created")
  out_folder1 = os.path.join(out_folder, "ImagePatch_SR/")
  out_folder2 = os.path.join(out_folder, "ImagePatch_HM/")

folder already created


In [2]:
gdf_ImagePatch = gpd.read_file(file_path)

print(len(gdf_ImagePatch)) # total lines in the AoI
print(gdf_ImagePatch.crs)  # current projection
gdf_ImagePatch.head(3)

In [1]:
working_folder = "01_Input_Data"
file_name = "AoI_id_082.geojson" # might be different for other areas

# file_path = project_root + task_folder + woring_folder + file_name
file_path = os.path.join(project_root, working_folder, file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
  gdf_ImageAoI = gpd.read_file(file_path)

  print(len(gdf_ImageAoI)) # total lines in the AoI
  print(gdf_ImageAoI.crs)  # current projection
  print(gdf_ImageAoI)
else:
  print("file missing")

In [ ]:
def test_image_date(date_min, date_max):
  booImageFound = False

  s2_sr = ee.ImageCollection('COPERNICUS/S2')\
    .filterDate(date_min, date_max)\
    .filterBounds(geometry) \
    .first() \

  try:
    #print(s2_sr.getInfo())
    sys_date = ee.Date(s2_sr.get('system:time_start'))

    time = sys_date.getInfo()
    #print(time.items())
    #print(time.get("value"))

    dt_time = datetime.datetime.fromtimestamp(time.get("value") // 1000)
    #print(dt_time)
    booImageFound = True

  except:
    #print("date is not valid, increasing window")
    booImageFound = False


  return booImageFound

In [ ]:
def getEVI(image):
    # Compute the EVI using an expression.
    EVI = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),
            'BLUE': image.select('B2').divide(10000)
        }).rename("B_EVI")

    image = image.addBands(EVI)

    return(image)

#   Calculate MSAVI2
def getMSAVI2(image):
    # Compute the MSAVI2 using an expression.
    MSAVI2 = image.expression(
        '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),

        }).rename("B_MSAVI2")

    image = image.addBands(MSAVI2)

    return(image)

def getNDMI(image):
    # Compute the NDMI using an expression.
    NDMI = image.expression(
        '(B8 - B11) / (B8 + B11)', {
            'B8': image.select('B8').divide(10000),
            'B11': image.select('B11').divide(10000),

        }).rename("B_NDMI")

    #alternative:
    #ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')

    image = image.addBands(NDMI)

    return(image)

def getNBR(image):
    # Compute the NBR using an expression.
    NBR = image.expression(
        '(NIR - SWIR) / (NIR + SWIR)', {
            'NIR': image.select('B8').divide(10000),
            'SWIR': image.select('B12').divide(10000),

        }).rename("B_NBR")

    #alternative:
    #nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')

    image = image.addBands(NBR)

    return(image)

#   calculated as  NDBI= (SWIR-NIR/(SWIR+NIR)
def getNDBI(image):
    # Compute the NDBI using an expression.
    NDBI = image.expression(
        '(SWIR -NIR) / (SWIR + NIR)', {
            'NIR': image.select('B8').divide(10000),
            'SWIR': image.select('B11').divide(10000),

        }).rename("B_NDBI")

    #alternative:
    #ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')

    image = image.addBands(NDBI)

    return(image)

def getNDWI(image):
    # Compute the MSAVI2 using an expression.
    NDWI = image.expression(
        '(GREEN - NIR) / (GREEN + NIR)', {
            'NIR': image.select('B8').divide(10000),
            'GREEN': image.select('B3').divide(10000),

        }).rename("B_NDWI")

    image = image.addBands(NDWI)

    #alternative:
    #ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    return(image)

#   calculated as NDVI = (NIR-RED) / (NIR+RED)

def getNDVI(image):
    # Compute the NDVI using an expression.
    NDVI = image.expression(
        '(NIR - RED) / (NIR + RED)', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),

        }).rename("B_NDVI")

    #alternative:
    #ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

    image = image.addBands(NDVI)

    return(image)

In [ ]:
def addDate(image):
    img_date = ee.Date(image.date())
    img_date = ee.Number.parse(img_date.format('YYYYMMdd'))
    return image.addBands(ee.Image(img_date).rename('date').toInt())

In [ ]:
def adjust_geometry_for_fixed_size_output(centroid, target_pixels, scale):
    """
    Adjusts the geometry around a centroid to ensure the exported image matches the target pixel dimensions.

    param centroid: Tuple of (longitude, latitude) for the AOI's centroid.
    param target_pixels: The target size in pixels (e.g., 128 for a 128x128 image).
    param scale: The scale in meters per pixel.

    return: An ee.Geometry.Rectangle object for the adjusted region.

    """

    # include ajustement factor
    # adjustment_factor = 0.9949 # Adjust this factor as needed to fine-tune the output size

    # Calculate the physical size of the target area in meters
    target_size_meters = target_pixels * scale

    # Convert meters to degrees approximately (1 degree = approx. 111,319.9 meters)
    target_size_degrees = target_size_meters / 111319.9

    # Calculate the bounds of the rectangle
    half_size_degrees = target_size_degrees / 2
    min_lon = centroid[0] - half_size_degrees
    max_lon = centroid[0] + half_size_degrees
    min_lat = centroid[1] - half_size_degrees
    max_lat = centroid[1] + half_size_degrees

    # Create and return the geometry
    return ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])



In [3]:
print("steps: ", len(gdf_ImagePatch))


In [ ]:
# Define output folders
folder_SR = out_folder1  # Sentinel Reflectance images
folder_HM = out_folder2  # Harmonized images

# Main loop over geodataframe entries
for k in tqdm_notebook(range(len(gdf_ImagePatch))):
    aoi = gdf_ImagePatch.iloc[k]
    strGID = aoi.GID

    # Construct file paths for potential output images
    file_path_SR = os.path.join(folder_SR, f"image_{strGID}.tif")
    file_path_HM = os.path.join(folder_HM, f"image_{strGID}.tif")

    # Check if the images already exist to skip processing
    if os.path.isfile(file_path_SR) and os.path.isfile(file_path_HM):
        print(f"Files for {strGID} already exist. Skipping processing.")
        continue

    # Proceed with calculations if files do not exist
    centroid = aoi.geometry.centroid.coords[0]
    fixed_pixels = 512
    scale = 10

    adjusted_geometry = adjust_geometry_for_fixed_size_output(
        centroid, target_pixels=fixed_pixels, scale=scale
    )

    minx, miny, maxx, maxy = aoi.geometry.bounds
    geometry = ee.Geometry.Rectangle([minx, miny, maxx, maxy])

    date_min = gdf_ImageAoI.min_date[0]
    date_max = gdf_ImageAoI.max_date[0]
    booTest = False  # Assume no valid image date is found initially

    for _ in range(10):  # Limit date testing to 10 iterations
        if test_image_date(date_min, date_max):
            booTest = True
            break

        # Adjust date_min and date_max by subtracting and adding one day respectively
        dt_object_min = datetime.datetime.strptime(date_min, '%Y-%m-%d') - datetime.timedelta(days=1)
        dt_object_max = datetime.datetime.strptime(date_max, '%Y-%m-%d') + datetime.timedelta(days=1)
        date_min = dt_object_min.strftime('%Y-%m-%d')
        date_max = dt_object_max.strftime('%Y-%m-%d')

    if booTest:
        # Processing image collections for both Sentinel Reflectance and Harmonized datasets
        collections = ['COPERNICUS/S2', 'COPERNICUS/S2_HARMONIZED']
        for collection, file_path in zip(collections, [file_path_SR, file_path_HM]):
            image_collection = ee.ImageCollection(collection) \
                .filterDate(date_min, date_max).filterBounds(adjusted_geometry) \
                .map(getNDVI).map(getEVI).map(getMSAVI2).map(getNDMI) \
                .map(getNBR).map(getNDBI).map(getNDWI).map(addDate).median()

            image_collection = image_collection.select('B.+')
            print(f"Processing {collection}: Number of bands after adding extra bands: {image_collection.bandNames().size().getInfo()}")

            try:
                geemap.ee_export_image(image_collection, filename=file_path, scale=10, region=adjusted_geometry.getInfo())
                print(image_collection.getInfo())
            except Exception as e:
                print(f"!!!!Issue!!!!.....check {strGID}........!!!!!!", e)

    else:
        print(f"No valid image found for {strGID}")

print("Processing complete")

  0%|          | 0/99 [00:00<?, ?it/s]

Processing COPERNICUS/S2: Number of bands after adding extra bands: 20
Generating URL ...
Please wait ...
Data downloaded to /Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_SB_010/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_SR/image_GID_SB_010_0_10.tif
{'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'double', 'mi